<center>
<img src="https://laelgelcpublic.s3.sa-east-1.amazonaws.com/lael_50_years_narrow_white.png.no_years.400px_96dpi.png" width="300" alt="LAEL 50 years logo">
<h3>APPLIED LINGUISTICS GRADUATE PROGRAMME (LAEL)</h3>
</center>
<hr>

# Corpus Linguistics - Study 1 - Phase 3 - Ednalvo - Evaluation of Compositions

## Setup

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

PROJECT_DIR = Path("/home/eyamrog/PycharmProjects/cl_st1_ednalvo/cl_st1_ph3_ednalvo")

ORIGINAL_SCORES_PATH = PROJECT_DIR / "corpus/00_fontes/composicoes_de_admissao_universitaria.tsv"
AI_ASSESSMENTS_DIR = PROJECT_DIR / "corpus/04_composicoes_avaliadas"
SAS_DIR = PROJECT_DIR / "sas"

SCORE_COLS = [
    "adequacao_ao_tema",
    "adequacao_a_coletanea",
    "adequacao_ao_tipo_de_texto",
    "adequacao_a_norma_padrao",
    "coesao",
    "coerencia",
]

TOTAL_COL = "pontuacao_total"

CANDIDATE_GROUP_ORDER = [
    "human_high",
    "human_low",
    "gemini_low_mirror",
    "gpt_low_mirror",
]

## Load and standardise human-assessed original scores

In [2]:
df_human_scores = pd.read_csv(ORIGINAL_SCORES_PATH, sep="\t")

df_human_scores["base_id"] = df_human_scores["inscricao"].astype(str)

df_human_scores["candidate_source"] = "human"
df_human_scores["candidate_band"] = df_human_scores["grupo"].map({
    "250_maiores_notas": "high",
    "250_menores_notas": "low",
})

df_human_scores["candidate_group"] = df_human_scores["candidate_band"].map({
    "high": "human_high",
    "low": "human_low",
})

df_human_scores["is_mirror"] = False
df_human_scores["mirror_model"] = pd.NA

df_human_scores["assessor_source"] = "human"
df_human_scores["assessor_model"] = pd.NA
df_human_scores["score_source_file"] = str(ORIGINAL_SCORES_PATH)

df_human_scores["composition_id"] = (
        df_human_scores["base_id"] + "_" + df_human_scores["candidate_group"]
)

df_human_scores = df_human_scores.rename(columns={
    "arquivo_original": "filename_original",
    "caminho_em_composicoes": "path",
})

for col in SCORE_COLS + [TOTAL_COL]:
    df_human_scores[col] = pd.to_numeric(df_human_scores[col], errors="coerce")

df_human_scores["score_sum_check"] = df_human_scores[SCORE_COLS].sum(axis=1)
df_human_scores["score_residual"] = (
        df_human_scores[TOTAL_COL] - df_human_scores["score_sum_check"]
)

df_human_scores.head()

,grupo,inscricao,filename_original,path,anotacoes,adequacao_ao_tema,adequacao_a_coletanea,adequacao_ao_tipo_de_texto,adequacao_a_norma_padrao,coesao,...,candidate_band,candidate_group,is_mirror,mirror_model,assessor_source,assessor_model,score_source_file,composition_id,score_sum_check,score_residual
0,250_maiores_notas,36237,36237 (0519-029).txt,corpus/01_composicoes/maiores_notas/36237.txt,{FO: inc findas},4.5,5.0,5.0,5.0,5.0,...,high,human_high,False,<NA>,human,<NA>,/home/eyamrog/PycharmProjects/cl_st1_ednalvo/c...,36237_human_high,29.5,0.0
1,250_maiores_notas,46656,46656 (0414-009).txt,corpus/01_composicoes/maiores_notas/46656.txt,NaN,5.0,5.0,4.0,4.5,5.0,...,high,human_high,False,<NA>,human,<NA>,/home/eyamrog/PycharmProjects/cl_st1_ednalvo/c...,46656_human_high,28.5,0.0
2,250_maiores_notas,36560,36560 (0502-031).txt,corpus/01_composicoes/maiores_notas/36560.txt,{FO: poluição}; {FO: más},4.5,4.5,4.5,5.0,5.0,...,high,human_high,False,<NA>,human,<NA>,/home/eyamrog/PycharmProjects/cl_st1_ednalvo/c...,36560_human_high,28.0,0.0
3,250_maiores_notas,36923,36923 (0416-024).txt,corpus/01_composicoes/maiores_notas/36923.txt,NaN,5.0,4.0,4.5,5.0,5.0,...,high,human_high,False,<NA>,human,<NA>,/home/eyamrog/PycharmProjects/cl_st1_ednalvo/c...,36923_human_high,28.0,0.0
4,250_maiores_notas,45355,45355 (0477-030).txt,corpus/01_composicoes/maiores_notas/45355.txt,"(terra, fogo, ar e água); (terra, fogo, ar e á...",4.5,4.5,4.5,5.0,5.0,...,high,human_high,False,<NA>,human,<NA>,/home/eyamrog/PycharmProjects/cl_st1_ednalvo/c...,45355_human_high,28.0,0.0


## Parse AI-assessment Markdown files

In [3]:
CRITERION_PATTERNS = {
    "adequacao_ao_tema": r"Adequa[cç][aã]o ao Tema",
    "adequacao_a_coletanea": r"Adequa[cç][aã]o [àa] Colet[âa]nea",
    "adequacao_ao_tipo_de_texto": r"Adequa[cç][aã]o ao tipo de texto",
    "adequacao_a_norma_padrao": r"Adequa[cç][aã]o [àa] norma padr[aã]o",
    "coesao": r"Coes[aã]o",
    "coerencia": r"Coer[êe]ncia",
    "pontuacao_total": r"Pontua[cç][aã]o Total",
}


def parse_ptbr_number(value: str) -> float:
    value = value.strip()
    value = value.replace("**", "")
    value = value.replace(",", ".")
    value = re.sub(r"[^0-9.\-]", "", value)
    return float(value) if value else np.nan


def extract_score_from_markdown(text: str, criterion_regex: str) -> float:
    pattern = rf"\|\s*[^|]*{criterion_regex}[^|]*\|\s*([^|]+?)\s*\|"
    match = re.search(pattern, text, flags=re.IGNORECASE)
    if not match:
        return np.nan
    return parse_ptbr_number(match.group(1))


def parse_ai_assessment_file(path: Path) -> dict:
    text = path.read_text(encoding="utf-8")

    base_id_match = re.search(r"(\d+)_avaliacao_", path.name)
    base_id = base_id_match.group(1) if base_id_match else pd.NA

    row = {
        "base_id": str(base_id),
        "ai_assessment_filename": path.name,
        "ai_assessment_path": str(path),
        "score_source_file": str(path),
    }

    for col, pattern in CRITERION_PATTERNS.items():
        row[col] = extract_score_from_markdown(text, pattern)

    return row

Quickly test on one file:

In [4]:
example_file = next((AI_ASSESSMENTS_DIR / "maiores_notas").glob("*_avaliacao_*.md"))
parse_ai_assessment_file(example_file)

{'base_id': '37662',
 'ai_assessment_filename': '37662_avaliacao_gpt-5.6-sol.md',
 'ai_assessment_path': '/home/eyamrog/PycharmProjects/cl_st1_ednalvo/cl_st1_ph3_ednalvo/corpus/04_composicoes_avaliadas/maiores_notas/37662_avaliacao_gpt-5.6-sol.md',
 'score_source_file': '/home/eyamrog/PycharmProjects/cl_st1_ednalvo/cl_st1_ph3_ednalvo/corpus/04_composicoes_avaliadas/maiores_notas/37662_avaliacao_gpt-5.6-sol.md',
 'adequacao_ao_tema': 4.0,
 'adequacao_a_coletanea': 4.0,
 'adequacao_ao_tipo_de_texto': 4.0,
 'adequacao_a_norma_padrao': 3.0,
 'coesao': 3.0,
 'coerencia': 4.0,
 'pontuacao_total': 22.0}

## Load all AI-assessed scores

In [5]:
AI_GROUP_MAP = {
    "maiores_notas": {
        "candidate_group": "human_high",
        "candidate_source": "human",
        "candidate_band": "high",
        "is_mirror": False,
        "mirror_model": pd.NA,
    },
    "menores_notas": {
        "candidate_group": "human_low",
        "candidate_source": "human",
        "candidate_band": "low",
        "is_mirror": False,
        "mirror_model": pd.NA,
    },
    "menores_notas_gemini": {
        "candidate_group": "gemini_low_mirror",
        "candidate_source": "gemini",
        "candidate_band": "low_mirror",
        "is_mirror": True,
        "mirror_model": "gemini",
    },
    "menores_notas_gpt": {
        "candidate_group": "gpt_low_mirror",
        "candidate_source": "gpt",
        "candidate_band": "low_mirror",
        "is_mirror": True,
        "mirror_model": "gpt",
    },
}


def load_ai_assessment_folder(folder_name: str, assessor_model: str = "gpt-5.6-sol") -> pd.DataFrame:
    folder = AI_ASSESSMENTS_DIR / folder_name
    files = sorted(folder.glob("*_avaliacao_*.md"))

    rows = [parse_ai_assessment_file(path) for path in files]
    df = pd.DataFrame(rows)

    meta = AI_GROUP_MAP[folder_name]
    for key, value in meta.items():
        df[key] = value

    df["assessor_source"] = "gpt_ai"
    df["assessor_model"] = assessor_model

    df["composition_id"] = df["base_id"] + "_" + df["candidate_group"]
    df["filename"] = df["base_id"] + ".txt"

    for col in SCORE_COLS + [TOTAL_COL]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["score_sum_check"] = df[SCORE_COLS].sum(axis=1)
    df["score_residual"] = df[TOTAL_COL] - df["score_sum_check"]

    return df


df_ai_scores = pd.concat(
    [
        load_ai_assessment_folder("maiores_notas"),
        load_ai_assessment_folder("menores_notas"),
        load_ai_assessment_folder("menores_notas_gemini"),
        load_ai_assessment_folder("menores_notas_gpt"),
    ],
    ignore_index=True,
)

df_ai_scores.head()

,base_id,ai_assessment_filename,ai_assessment_path,score_source_file,adequacao_ao_tema,adequacao_a_coletanea,adequacao_ao_tipo_de_texto,adequacao_a_norma_padrao,coesao,coerencia,...,candidate_source,candidate_band,is_mirror,mirror_model,assessor_source,assessor_model,composition_id,filename,score_sum_check,score_residual
0,32757,32757_avaliacao_gpt-5.6-sol.md,/home/eyamrog/PycharmProjects/cl_st1_ednalvo/c...,/home/eyamrog/PycharmProjects/cl_st1_ednalvo/c...,2.5,2.5,3.5,3.0,3.5,4.0,...,human,high,False,<NA>,gpt_ai,gpt-5.6-sol,32757_human_high,32757.txt,19.0,0.0
1,32875,32875_avaliacao_gpt-5.6-sol.md,/home/eyamrog/PycharmProjects/cl_st1_ednalvo/c...,/home/eyamrog/PycharmProjects/cl_st1_ednalvo/c...,3.0,3.0,5.0,2.0,4.0,5.0,...,human,high,False,<NA>,gpt_ai,gpt-5.6-sol,32875_human_high,32875.txt,22.0,0.0
2,33083,33083_avaliacao_gpt-5.6-sol.md,/home/eyamrog/PycharmProjects/cl_st1_ednalvo/c...,/home/eyamrog/PycharmProjects/cl_st1_ednalvo/c...,2.5,2.5,3.5,4.5,3.5,3.5,...,human,high,False,<NA>,gpt_ai,gpt-5.6-sol,33083_human_high,33083.txt,20.0,0.0
3,33228,33228_avaliacao_gpt-5.6-sol.md,/home/eyamrog/PycharmProjects/cl_st1_ednalvo/c...,/home/eyamrog/PycharmProjects/cl_st1_ednalvo/c...,3.0,3.0,4.0,0.0,4.0,4.0,...,human,high,False,<NA>,gpt_ai,gpt-5.6-sol,33228_human_high,33228.txt,18.0,0.0
4,33383,33383_avaliacao_gpt-5.6-sol.md,/home/eyamrog/PycharmProjects/cl_st1_ednalvo/c...,/home/eyamrog/PycharmProjects/cl_st1_ednalvo/c...,3.0,3.0,4.0,2.0,3.5,4.0,...,human,high,False,<NA>,gpt_ai,gpt-5.6-sol,33383_human_high,33383.txt,19.5,0.0


## Build the canonical long-format score table

In [6]:
COMMON_SCORE_COLS = [
    "composition_id",
    "base_id",
    "candidate_source",
    "candidate_group",
    "candidate_band",
    "is_mirror",
    "mirror_model",
    "assessor_source",
    "assessor_model",
    "path",
    "filename",
    "filename_original",
    "ai_assessment_filename",
    "ai_assessment_path",
    "score_source_file",
    *SCORE_COLS,
    TOTAL_COL,
    "score_sum_check",
    "score_residual",
]

for col in COMMON_SCORE_COLS:
    if col not in df_human_scores.columns:
        df_human_scores[col] = pd.NA
    if col not in df_ai_scores.columns:
        df_ai_scores[col] = pd.NA

df_scores_long = pd.concat(
    [
        df_human_scores[COMMON_SCORE_COLS],
        df_ai_scores[COMMON_SCORE_COLS],
    ],
    ignore_index=True,
)

for col in SCORE_COLS + [TOTAL_COL, "score_sum_check", "score_residual"]:
    df_scores_long[col] = pd.to_numeric(df_scores_long[col], errors="coerce")

df_scores_long["candidate_group"] = pd.Categorical(
    df_scores_long["candidate_group"],
    categories=CANDIDATE_GROUP_ORDER,
    ordered=True,
)

df_scores_long.head()

,composition_id,base_id,candidate_source,candidate_group,candidate_band,is_mirror,mirror_model,assessor_source,assessor_model,path,...,score_source_file,adequacao_ao_tema,adequacao_a_coletanea,adequacao_ao_tipo_de_texto,adequacao_a_norma_padrao,coesao,coerencia,pontuacao_total,score_sum_check,score_residual
0,36237_human_high,36237,human,human_high,high,False,<NA>,human,<NA>,corpus/01_composicoes/maiores_notas/36237.txt,...,/home/eyamrog/PycharmProjects/cl_st1_ednalvo/c...,4.5,5.0,5.0,5.0,5.0,5.0,29.5,29.5,0.0
1,46656_human_high,46656,human,human_high,high,False,<NA>,human,<NA>,corpus/01_composicoes/maiores_notas/46656.txt,...,/home/eyamrog/PycharmProjects/cl_st1_ednalvo/c...,5.0,5.0,4.0,4.5,5.0,5.0,28.5,28.5,0.0
2,36560_human_high,36560,human,human_high,high,False,<NA>,human,<NA>,corpus/01_composicoes/maiores_notas/36560.txt,...,/home/eyamrog/PycharmProjects/cl_st1_ednalvo/c...,4.5,4.5,4.5,5.0,5.0,4.5,28.0,28.0,0.0
3,36923_human_high,36923,human,human_high,high,False,<NA>,human,<NA>,corpus/01_composicoes/maiores_notas/36923.txt,...,/home/eyamrog/PycharmProjects/cl_st1_ednalvo/c...,5.0,4.0,4.5,5.0,5.0,4.5,28.0,28.0,0.0
4,45355_human_high,45355,human,human_high,high,False,<NA>,human,<NA>,corpus/01_composicoes/maiores_notas/45355.txt,...,/home/eyamrog/PycharmProjects/cl_st1_ednalvo/c...,4.5,4.5,4.5,5.0,5.0,4.5,28.0,28.0,0.0


Conceptually:
- `df_scores_long` has one row per composition × assessor.
- The original human essays should have human and AI rows.
- The LLM mirrored essays should have AI rows only.

## Validation checks

In [7]:
print("Rows by candidate group and assessor:")
display(
    df_scores_long
    .groupby(["candidate_group", "assessor_source"], observed=False)
    .size()
    .reset_index(name="n")
)

print("Unique compositions by candidate group:")
display(
    df_scores_long[["composition_id", "candidate_group"]]
    .drop_duplicates()
    .groupby("candidate_group", observed=False)
    .size()
    .reset_index(name="n_compositions")
)

print("Score residual summary:")
display(df_scores_long["score_residual"].describe())

print("Missing AI score fields:")
display(df_ai_scores[SCORE_COLS + [TOTAL_COL]].isna().sum())

print("Duplicate assessment rows:")
duplicate_count = df_scores_long.duplicated(
    subset=["composition_id", "assessor_source", "assessor_model"]
).sum()
display(duplicate_count)

Rows by candidate group and assessor:


,candidate_group,assessor_source,n
0,human_high,gpt_ai,250
1,human_high,human,250
2,human_low,gpt_ai,250
3,human_low,human,250
4,gemini_low_mirror,gpt_ai,250
5,gemini_low_mirror,human,0
6,gpt_low_mirror,gpt_ai,250
7,gpt_low_mirror,human,0


Unique compositions by candidate group:


,candidate_group,n_compositions
0,human_high,250
1,human_low,250
2,gemini_low_mirror,250
3,gpt_low_mirror,250


Score residual summary:


count    1500.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: score_residual, dtype: float64

Missing AI score fields:


adequacao_ao_tema             0
adequacao_a_coletanea         0
adequacao_ao_tipo_de_texto    0
adequacao_a_norma_padrao      0
coesao                        0
coerencia                     0
pontuacao_total               0
dtype: int64

Duplicate assessment rows:


np.int64(0)